# LeetCode #19 — Remove Nth Node From End of List: hai con trỏ cách nhau `n` bước

**Mức độ:** Medium · **Chủ đề:** Linked List, Two Pointers

Skeleton trên LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def removeNthFromEnd(self, head: Optional[ListNode], n: int) -> Optional[ListNode]:
```

Mục tiêu của notebook:

1. Hiểu cách **xóa một nút** khỏi Linked List: phải đứng ở **nút ngay trước** nó.
2. Dùng **dummy node** để xóa được cả nút đầu mà không cần `if` riêng.
3. Kỹ thuật **hai con trỏ nhanh/chậm giữ khoảng cách cố định** — giải bài trong **một lần duyệt**.

# 1. Bài toán là gì?

Cho `head` của một Linked List. Xóa nút thứ `n` **tính từ cuối** danh sách và trả về `head` mới.

**Constraints (theo đề):**

- Số nút là `sz`, `1 <= sz <= 30`
- `0 <= Node.val <= 100`
- `1 <= n <= sz`

**Follow up:** bạn có làm được trong **một lần duyệt** không?

---

## Ví dụ

```text
head = [1, 2, 3, 4, 5], n = 2  ->  [1, 2, 3, 5]     (nút thứ 2 từ cuối là 4)
head = [1], n = 1              ->  []
head = [1, 2], n = 1           ->  [1]
```

Đếm từ cuối: nút cuối là thứ 1, nút áp chót là thứ 2, …

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)
2. Hai con trỏ (two pointers)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [2]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [3]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [4]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [5]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [6]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [7]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.3 Áp dụng vào bài này

Muốn xóa nút thứ `n` **từ cuối** trong một lượt: cho con trỏ `fast` đi trước `n` bước, rồi `fast` và `slow` cùng đi — khi `fast` tới cuối, `slow` đứng **ngay trước** nút cần xóa (kỹ thuật hai con trỏ cách nhau một khoảng cố định). Xóa bằng `slow.next = slow.next.next`; **nút giả** xử lý trường hợp xóa chính `head`.

# 3. Nền tảng: Linked List và cách xóa một nút

Nhắc lại `ListNode` và hai hàm tiện ích (đã gặp ở bài #2 Add Two Numbers):

In [8]:
from typing import Optional, List


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values: List[int]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head: Optional[ListNode]) -> List[int]:
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


print(to_list(build_list([1, 2, 3, 4, 5])))

[1, 2, 3, 4, 5]

## Xóa nút: "nối vượt qua" nó

Muốn xóa nút `X` trong `A -> X -> B`, ta **không** đứng ở `X`. Ta đứng ở `A` (nút **trước** `X`) và cho `A` trỏ thẳng tới `B`:

```python
A.next = A.next.next
```

```text
trước:  A -> X -> B
sau:    A ------> B        (X không còn ai trỏ tới, coi như đã xóa)
```

Vì Linked List chỉ đi một chiều, **đứng ở `X` thì không sửa được mũi tên của `A`**. Đây là lý do mọi thao tác xóa đều cần **con trỏ tới nút đứng trước**.

In [9]:
head = build_list([1, 2, 3, 4, 5])
prev = head.next.next          # nút 3, đứng trước nút 4
prev.next = prev.next.next     # bỏ qua nút 4
print(to_list(head))

[1, 2, 3, 5]


## Vấn đề: xóa nút **đầu tiên**

Nút đầu không có nút nào đứng trước! Với `[1, 2]`, `n = 2` (xóa nút 1), ta phải trả về `head.next` — một trường hợp riêng.

**Dummy node** giải quyết gọn: thêm một nút giả trước `head`. Khi đó **mọi** nút thật, kể cả nút đầu, đều có nút đứng trước.

```text
dummy -> 1 -> 2 -> None
```

Xóa nút 1: đứng ở `dummy`, `dummy.next = dummy.next.next`. Trả về `dummy.next`.

In [10]:
head = build_list([1, 2])
dummy = ListNode(0, head)
dummy.next = dummy.next.next     # xóa nút đầu
print(to_list(dummy.next))

[2]


# 4. Cách 1: hai lần duyệt (đếm độ dài)

Nút thứ `n` từ cuối = nút thứ `L - n + 1` từ đầu (đánh số từ 1), với `L` là độ dài.

Vậy nút **đứng trước** nó là nút thứ `L - n` từ đầu. Tính từ `dummy` (vị trí 0), ta đi `L - n` bước.

1. Lần 1: đếm `L`.
2. Lần 2: từ `dummy` đi `L - n` bước, rồi xóa.

Ví dụ `[1, 2, 3, 4, 5]`, `n = 2`: `L = 5`, đi `3` bước từ dummy → tới nút `3`, xóa nút `4`.

In [11]:
def remove_nth_two_pass(head: Optional[ListNode], n: int) -> Optional[ListNode]:
    length = 0
    node = head
    while node:
        length += 1
        node = node.next

    dummy = ListNode(0, head)
    prev = dummy
    for _ in range(length - n):
        prev = prev.next
    prev.next = prev.next.next
    return dummy.next

print(to_list(remove_nth_two_pass(build_list([1, 2, 3, 4, 5]), 2)))
print(to_list(remove_nth_two_pass(build_list([1]), 1)))
print(to_list(remove_nth_two_pass(build_list([1, 2]), 1)))
print(to_list(remove_nth_two_pass(build_list([1, 2]), 2)))

[1, 2, 3, 5]
[]
[1]
[2]


Cách này **đúng** và `O(L)`. Nhưng follow-up yêu cầu **một lần duyệt**. Ta không biết `L` trước — làm sao biết lúc nào dừng?

# 5. Ý tưởng chính: hai con trỏ cách nhau `n` bước

Hãy tưởng tượng hai người đi trên danh sách, người `fast` đi trước người `slow` **đúng `n + 1` bước**. Hai người đi **cùng tốc độ**.

Khi `fast` ra khỏi danh sách (thành `None`), `slow` cách cuối danh sách đúng `n + 1` bước — tức `slow` đứng **ngay trước** nút thứ `n` từ cuối!

```text
dummy -> 1 -> 2 -> 3 -> 4 -> 5 -> None        n = 2
```

1. `slow = fast = dummy`.
2. Cho `fast` đi trước `n + 1 = 3` bước: `fast` ở nút `3`.
3. Cùng đi tới khi `fast` là `None`:

| fast | slow |
|---|---|
| 3 | dummy |
| 4 | 1 |
| 5 | 2 |
| None | **3** |

4. `slow` ở nút `3`, đứng trước nút `4` (thứ 2 từ cuối). Xóa: `slow.next = slow.next.next`.

**Vì sao `n + 1` mà không phải `n`?** Ta muốn `slow` dừng ở nút **trước** nút cần xóa, nên khoảng cách phải lớn hơn 1.

In [12]:
def remove_nth_from_end(head: Optional[ListNode], n: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    fast = slow = dummy

    for _ in range(n + 1):          # fast đi trước n + 1 bước
        fast = fast.next

    while fast:                     # cùng đi tới khi fast ra khỏi danh sách
        fast = fast.next
        slow = slow.next

    slow.next = slow.next.next      # slow đứng ngay trước nút cần xóa
    return dummy.next

print(to_list(remove_nth_from_end(build_list([1, 2, 3, 4, 5]), 2)))
print(to_list(remove_nth_from_end(build_list([1]), 1)))
print(to_list(remove_nth_from_end(build_list([1, 2]), 1)))
print(to_list(remove_nth_from_end(build_list([1, 2]), 2)))

[1, 2, 3, 5]
[]
[1]
[2]


`n <= sz` nên trong `n + 1` bước đầu, `fast` đi tối đa tới `None` (khi `n = sz`) — không bao giờ gọi `.next` trên `None`. Nhờ có `dummy`, độ dài thật là `sz + 1`.

# 6. Mô phỏng từng bước

In [13]:
def trace_remove_nth(values: List[int], n: int):
    def name(node, dummy):
        if node is None:
            return "None"
        return "dummy" if node is dummy else str(node.val)

    head = build_list(values)
    dummy = ListNode(0, head)
    fast = slow = dummy
    print(f"Danh sách: {values}, n = {n}")

    for step in range(n + 1):
        fast = fast.next
        print(f"  fast đi trước bước {step + 1}: fast = {name(fast, dummy)}")

    while fast:
        fast = fast.next
        slow = slow.next
        print(f"  cùng đi: fast = {name(fast, dummy):>5}, slow = {name(slow, dummy)}")

    print(f"  slow đứng ở {name(slow, dummy)}, xóa nút {slow.next.val}")
    slow.next = slow.next.next
    print("  Kết quả:", to_list(dummy.next))

trace_remove_nth([1, 2, 3, 4, 5], 2)
print()
trace_remove_nth([1, 2], 2)

Danh sách: [1, 2, 3, 4, 5], n = 2
  fast đi trước bước 1: fast = 1
  fast đi trước bước 2: fast = 2
  fast đi trước bước 3: fast = 3
  cùng đi: fast =     4, slow = 1
  cùng đi: fast =     5, slow = 2
  cùng đi: fast =  None, slow = 3
  slow đứng ở 3, xóa nút 4
  Kết quả: [1, 2, 3, 5]

Danh sách: [1, 2], n = 2
  fast đi trước bước 1: fast = 1
  fast đi trước bước 2: fast = 2
  fast đi trước bước 3: fast = None
  slow đứng ở dummy, xóa nút 1
  Kết quả: [2]


Ví dụ thứ hai xóa **nút đầu**: `fast` đi 3 bước tới `None` ngay, vòng `while` không chạy, `slow` vẫn là `dummy` → xóa nút sau dummy (nút `1`). Không cần `if` riêng nhờ dummy node.

# 7. Phiên bản đúng format LeetCode

In [14]:
class Solution:
    def removeNthFromEnd(self, head: Optional[ListNode], n: int) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        fast = slow = dummy
        for _ in range(n + 1):
            fast = fast.next
        while fast:
            fast = fast.next
            slow = slow.next
        slow.next = slow.next.next
        return dummy.next

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS  {actual}")


sol = Solution()
run = lambda vals, n: to_list(sol.removeNthFromEnd(build_list(vals), n))

check("Ví dụ 1", run([1, 2, 3, 4, 5], 2), [1, 2, 3, 5])
check("Ví dụ 2: một nút", run([1], 1), [])
check("Ví dụ 3", run([1, 2], 1), [1])
check("Xóa nút đầu", run([1, 2], 2), [2])
check("Xóa nút cuối", run([1, 2, 3], 1), [1, 2])
check("Xóa nút đầu (dài)", run([1, 2, 3, 4, 5], 5), [2, 3, 4, 5])
check("Giá trị trùng", run([7, 7, 7], 2), [7, 7])

Ví dụ 1                    PASS  [1, 2, 3, 5]


Ví dụ 2: một nút           PASS  []
Ví dụ 3                    PASS  [1]
Xóa nút đầu                PASS  [2]
Xóa nút cuối               PASS  [1, 2]
Xóa nút đầu (dài)          PASS  [2, 3, 4, 5]
Giá trị trùng              PASS  [7, 7]


In [16]:
import random

random.seed(19)
for _ in range(3000):
    vals = [random.randint(0, 100) for _ in range(random.randint(1, 30))]
    n = random.randint(1, len(vals))
    expected = vals[:len(vals) - n] + vals[len(vals) - n + 1:]
    assert run(vals, n) == expected, (vals, n)
    assert to_list(remove_nth_two_pass(build_list(vals), n)) == expected
print("3000 test ngẫu nhiên: cả hai cách đều đúng")

3000 test ngẫu nhiên: cả hai cách đều đúng

# 8. Vì sao đúng? Độ phức tạp?

**Bất biến:** sau bước đi trước, `fast` đứng trước `slow` đúng `n + 1` nút. Hai con trỏ đi cùng tốc độ nên khoảng cách này **không đổi**.

Khi `fast == None` (vị trí "ngay sau nút cuối"), `slow` ở vị trí cách đó `n + 1` → `slow.next` là nút thứ `n` tính từ cuối. Xóa `slow.next` là đúng yêu cầu.

**Độ phức tạp** (`L` là số nút):

```text
Time:  O(L)   — một lần duyệt (fast đi L + 1 bước)
Space: O(1)
```

| Cách | Số lần duyệt | Thời gian | Bộ nhớ |
|---|---|---|---|
| Đếm độ dài | 2 | O(L) | O(1) |
| Hai con trỏ | 1 | O(L) | O(1) |
| Lưu mọi nút vào list | 1 | O(L) | O(L) |

# 9. Những lỗi dễ mắc

## Lỗi 1: Không dùng dummy, quên trường hợp xóa nút đầu

In [17]:
def wrong_no_dummy(head: Optional[ListNode], n: int) -> Optional[ListNode]:
    fast = slow = head
    for _ in range(n):
        fast = fast.next
    while fast and fast.next:
        fast = fast.next
        slow = slow.next
    slow.next = slow.next.next            # khi cần xóa head, slow.next là nút SAU head
    return head

print("Sai :", to_list(wrong_no_dummy(build_list([1, 2, 3]), 3)))
print("Đúng:", to_list(remove_nth_from_end(build_list([1, 2, 3]), 3)))

Sai : [1, 3]
Đúng: [2, 3]


Code không dummy cần thêm: `if fast is None: return head.next` sau bước đi trước. Dummy node loại bỏ hẳn nhánh này.

## Lỗi 2: Cho `fast` đi trước `n` bước thay vì `n + 1` (khi bắt đầu từ dummy)

`slow` sẽ dừng **ngay tại** nút cần xóa thay vì trước nó → xóa nhầm nút kế tiếp.

In [18]:
def wrong_gap_n(head: Optional[ListNode], n: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    fast = slow = dummy
    for _ in range(n):                    # SAI: thiếu 1 bước
        fast = fast.next
    while fast:
        fast = fast.next
        slow = slow.next
    if slow.next:
        slow.next = slow.next.next
    return dummy.next

print("Sai :", to_list(wrong_gap_n(build_list([1, 2, 3, 4, 5]), 2)))
print("Đúng:", to_list(remove_nth_from_end(build_list([1, 2, 3, 4, 5]), 2)))

Sai : [1, 2, 3, 4]
Đúng: [1, 2, 3, 5]


## Lỗi 3: Trả về `head` thay vì `dummy.next`

Nếu nút đầu bị xóa, `head` vẫn trỏ tới nút đã xóa. Luôn trả `dummy.next`.

## Lỗi 4: Đứng ở nút cần xóa rồi cố xóa nó

Đứng ở `X` thì không sửa được `A.next`. (Có mẹo "chép giá trị nút sau vào `X` rồi xóa nút sau" — bài **#237** — nhưng mẹo đó không xóa được nút cuối.)

# 10. Mẹo nhận diện pattern

**Hai con trỏ cùng tốc độ, cách nhau cố định** khi đề hỏi vị trí "thứ `k` từ cuối" trên cấu trúc chỉ đi một chiều:

- **#19** Remove Nth Node From End
- **#61** Rotate List (tìm nút thứ k từ cuối để cắt)
- "Return kth to last" (Cracking the Coding Interview)

**Hai con trỏ khác tốc độ (nhanh gấp đôi)** là họ hàng gần:

- **#876** Middle of the Linked List
- **#141 / #142** Linked List Cycle

**Dummy node** khi thao tác có thể **thay đổi nút đầu** (xóa, chèn, đảo).

## Template

```python
dummy = ListNode(0, head)
fast = slow = dummy
for _ in range(k + 1):
    fast = fast.next
while fast:
    fast, slow = fast.next, slow.next
# slow đứng ngay trước nút thứ k từ cuối
```

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `[10, 20, 30, 40, 50, 60]` và `n = 4`, lập bảng vị trí `fast` / `slow` qua từng bước. Nút nào bị xóa?

## Bài 2 — Giá trị thứ `k` từ cuối

Trả về **giá trị** của nút thứ `k` từ cuối (không xóa), một lần duyệt.

```python
kth_from_end(build_list([1, 2, 3, 4, 5]), 1) -> 5
kth_from_end(build_list([1, 2, 3, 4, 5]), 5) -> 1
```

In [19]:
def exercise_2_kth_from_end(head: Optional[ListNode], k: int) -> int:
    # TODO: fast đi trước k bước (bắt đầu từ head), rồi cùng đi tới khi fast là None
    pass


# assert exercise_2_kth_from_end(build_list([1, 2, 3, 4, 5]), 1) == 5
# assert exercise_2_kth_from_end(build_list([1, 2, 3, 4, 5]), 5) == 1
# assert exercise_2_kth_from_end(build_list([1, 2, 3, 4, 5]), 2) == 4

## Bài 3 — Điền chỗ trống

In [20]:
def exercise_3_remove_nth(head: Optional[ListNode], n: int) -> Optional[ListNode]:
    # TODO 1: tạo dummy trỏ tới head, fast = slow = dummy
    # TODO 2: fast đi trước n + 1 bước
    # TODO 3: cùng đi tới khi fast là None
    # TODO 4: xóa slow.next, trả về dummy.next
    pass


# assert to_list(exercise_3_remove_nth(build_list([1, 2, 3, 4, 5]), 2)) == [1, 2, 3, 5]
# assert to_list(exercise_3_remove_nth(build_list([1]), 1)) == []
# assert to_list(exercise_3_remove_nth(build_list([1, 2]), 2)) == [2]

## Bài 4 — Biến thể: nút ở giữa (LeetCode #876)

Trả về nút ở giữa. Nếu có hai nút giữa, trả về nút **thứ hai**. Dùng `slow` đi 1 bước, `fast` đi 2 bước.

```python
middle([1, 2, 3, 4, 5])    -> nút 3
middle([1, 2, 3, 4, 5, 6]) -> nút 4
```

In [21]:
def exercise_4_middle_node(head: Optional[ListNode]) -> Optional[ListNode]:
    # TODO
    pass


# assert exercise_4_middle_node(build_list([1, 2, 3, 4, 5])).val == 3
# assert exercise_4_middle_node(build_list([1, 2, 3, 4, 5, 6])).val == 4
# assert exercise_4_middle_node(build_list([1])).val == 1

## Bài 5 — Mini challenge: Rotate List (LeetCode #61)

Xoay danh sách sang phải `k` vị trí. `k` có thể lớn hơn độ dài.

```text
[1, 2, 3, 4, 5], k = 2  ->  [4, 5, 1, 2, 3]
[0, 1, 2],       k = 4  ->  [2, 0, 1]
[],              k = 1  ->  []
```

Gợi ý nhanh: xoay `k` bước = cắt ở nút thứ `k mod L` từ cuối và nối phần sau lên đầu.

In [22]:
def exercise_5_rotate_right(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_5_rotate_right(build_list([1, 2, 3, 4, 5]), 2)) == [4, 5, 1, 2, 3]
# assert to_list(exercise_5_rotate_right(build_list([0, 1, 2]), 4)) == [2, 0, 1]
# assert to_list(exercise_5_rotate_right(build_list([]), 1)) == []
# assert to_list(exercise_5_rotate_right(build_list([1, 2]), 2)) == [1, 2]

# 12. Gợi ý

**Bài 1**
- Mức 1: `fast` đi trước `n + 1 = 5` bước từ dummy.
- Mức 2: khi đó `fast` ở nút `50`.
- Mức 3: thêm 2 bước nữa `fast` thành `None`.

**Bài 2**
- Mức 1: không cần dummy vì không xóa.
- Mức 2: bắt đầu `fast = slow = head`, `fast` đi trước `k` bước; khi `fast` là `None`, `slow` chính là nút cần tìm.

**Bài 4**
- Mức 1: `while fast and fast.next:`.
- Mức 2: mỗi vòng `slow = slow.next`, `fast = fast.next.next`.

**Bài 5**
- Mức 1: đếm độ dài `L` và tìm nút cuối `tail` trong cùng một lần duyệt.
- Mức 2: `k %= L`; nếu `k == 0` trả `head`.
- Mức 3: nút mới cuối là nút thứ `L - k` (từ 1); nối `tail.next = head`, cắt sau nút mới cuối.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [23]:
trace_remove_nth([10, 20, 30, 40, 50, 60], 4)

Danh sách: [10, 20, 30, 40, 50, 60], n = 4
  fast đi trước bước 1: fast = 10
  fast đi trước bước 2: fast = 20
  fast đi trước bước 3: fast = 30
  fast đi trước bước 4: fast = 40
  fast đi trước bước 5: fast = 50
  cùng đi: fast =    60, slow = 10
  cùng đi: fast =  None, slow = 20
  slow đứng ở 20, xóa nút 30
  Kết quả: [10, 20, 40, 50, 60]


## Đáp án Bài 2

In [24]:
def answer_2_kth_from_end(head: Optional[ListNode], k: int) -> int:
    fast = slow = head
    for _ in range(k):
        fast = fast.next
    while fast:
        fast = fast.next
        slow = slow.next
    return slow.val

assert answer_2_kth_from_end(build_list([1, 2, 3, 4, 5]), 1) == 5
assert answer_2_kth_from_end(build_list([1, 2, 3, 4, 5]), 5) == 1
assert answer_2_kth_from_end(build_list([1, 2, 3, 4, 5]), 2) == 4
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [25]:
def answer_3_remove_nth(head: Optional[ListNode], n: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    fast = slow = dummy
    for _ in range(n + 1):
        fast = fast.next
    while fast:
        fast = fast.next
        slow = slow.next
    slow.next = slow.next.next
    return dummy.next

assert to_list(answer_3_remove_nth(build_list([1, 2, 3, 4, 5]), 2)) == [1, 2, 3, 5]
assert to_list(answer_3_remove_nth(build_list([1]), 1)) == []
assert to_list(answer_3_remove_nth(build_list([1, 2]), 2)) == [2]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [26]:
def answer_4_middle_node(head: Optional[ListNode]) -> Optional[ListNode]:
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

assert answer_4_middle_node(build_list([1, 2, 3, 4, 5])).val == 3
assert answer_4_middle_node(build_list([1, 2, 3, 4, 5, 6])).val == 4
assert answer_4_middle_node(build_list([1])).val == 1
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [27]:
def answer_5_rotate_right(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    if not head:
        return head
    length, tail = 1, head
    while tail.next:
        tail = tail.next
        length += 1
    k %= length
    if k == 0:
        return head
    new_tail = head
    for _ in range(length - k - 1):
        new_tail = new_tail.next
    new_head = new_tail.next
    new_tail.next = None
    tail.next = head
    return new_head

assert to_list(answer_5_rotate_right(build_list([1, 2, 3, 4, 5]), 2)) == [4, 5, 1, 2, 3]
assert to_list(answer_5_rotate_right(build_list([0, 1, 2]), 4)) == [2, 0, 1]
assert to_list(answer_5_rotate_right(build_list([]), 1)) == []
assert to_list(answer_5_rotate_right(build_list([1, 2]), 2)) == [1, 2]
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Xóa nút cần **đứng ở nút trước nó**. Dummy node đảm bảo nút nào cũng có "nút trước". Hai con trỏ cách nhau `n + 1` bước đưa `slow` tới đúng chỗ trong **một lần duyệt**.

---

## Checklist trước khi nộp

- [ ] Dùng `dummy = ListNode(0, head)`?
- [ ] `fast` đi trước **`n + 1`** bước (bắt đầu từ dummy)?
- [ ] Vòng `while fast:` di chuyển **cả hai** con trỏ?
- [ ] Trả về `dummy.next`?
- [ ] Đã thử xóa nút đầu, nút cuối, danh sách 1 nút?